### Install dependencies

In [1]:
# TotalSegmentator >= 2.x includes the liver_segments and liver_lesions tasks
# (liver_lesions / liver_lesions_mr cite this exact paper in the README).
# !pip install -q TotalSegmentator nibabel numpy pandas matplotlib


In [2]:
# !pip install nibabel


### Configuration



In [1]:
from pathlib import Path

INPUT_DIR = Path("CT_scans")
OUTPUT_DIR = Path("liver_seg_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cases = []

for class_dir in INPUT_DIR.iterdir():
    if class_dir.is_dir():

        (OUTPUT_DIR / class_dir.name).mkdir(exist_ok=True)

        nii_files = sorted(class_dir.glob("*.nii.gz")) + sorted(class_dir.glob("*.nii"))

        for f in nii_files:
            cases.append({
                "class": class_dir.name,
                "path": f
            })

print(f"Found {len(cases)} cases.")

Found 60 cases.


In [2]:
print(cases[0])

{'class': 'abnormal_liver_and_other', 'path': WindowsPath('CT_scans/abnormal_liver_and_other/AC4213693.nii.gz')}



### Models

For every case we run:
- `liver_segments` 
- `liver_lesions` 


In [5]:
# totalsegmentator(
#     input=str(input_path),
#     output=str(task_out),
#     task=task,
#     statistics=True,
#     quiet=True,
#     nr_thr_resamp=1,
#     nr_thr_saving=1,
# )

In [4]:
import json
import traceback
from pathlib import Path
from totalsegmentator.python_api import totalsegmentator


def run_task(input_path: Path, out_dir: Path, task: str) -> dict:

    task_out = out_dir / task
    task_out.mkdir(parents=True, exist_ok=True)

    stats_file = task_out / "statistics.json"

    # Skip if already processed
    if stats_file.exists():
        print(f"    -> {task} already exists. Skipping.")
        with open(stats_file) as f:
            return json.load(f)

    # Run TotalSegmentator
    totalsegmentator(
        input=str(input_path),
        output=str(task_out),
        task=task,
        statistics=True,
        quiet=True,
        nr_thr_resamp=1,
        nr_thr_saving=1,
    )

    # Read statistics
    if stats_file.exists():
        with open(stats_file) as f:
            return json.load(f)

    return {}


results = {}

for case in cases:

    input_path = case["path"]
    class_name = case["class"]
    case_name = input_path.stem.replace(".nii", "")

    case_out = OUTPUT_DIR / class_name / case_name
    case_out.mkdir(parents=True, exist_ok=True)

    print(f"\n=== [{class_name}] {case_name} ===")

    results[case_name] = {
        "class": class_name
    }

    for task in ["liver_segments", "liver_lesions"]:

        try:
            print(f"  Task: {task}")

            stats = run_task(input_path, case_out, task)
            results[case_name][task] = stats

        except Exception as e:

            print(f"  FAILED on task {task}: {e}")
            traceback.print_exc()

            results[case_name][task] = None

print("\nDone.")


=== [abnormal_liver_and_other] AC4213693 ===
  Task: liver_segments
    -> liver_segments already exists. Skipping.
  Task: liver_lesions
    -> liver_lesions already exists. Skipping.

=== [abnormal_liver_and_other] AC4213725 ===
  Task: liver_segments
    -> liver_segments already exists. Skipping.
  Task: liver_lesions
    -> liver_lesions already exists. Skipping.

=== [abnormal_liver_and_other] AC42137eb ===
  Task: liver_segments
    -> liver_segments already exists. Skipping.
  Task: liver_lesions
    -> liver_lesions already exists. Skipping.

=== [abnormal_liver_and_other] AC4213995 ===
  Task: liver_segments
    -> liver_segments already exists. Skipping.
  Task: liver_lesions
    -> liver_lesions already exists. Skipping.

=== [abnormal_liver_and_other] AC4213e2b ===
  Task: liver_segments
    -> liver_segments already exists. Skipping.
  Task: liver_lesions
    -> liver_lesions already exists. Skipping.

=== [abnormal_liver_and_other] AC4213e68 ===
  Task: liver_segments
 


### Summary 



In [ ]:
import pandas as pd

SEGMENT_CLASSES = [f"liver_segment_{i}" for i in range(1, 9)]

rows = []

for case_name, task_results in results.items():

    row = {
        "class": task_results.get("class"),
        "case": case_name
    }

    # Liver segment statistics
    seg_stats = task_results.get("liver_segments") or {}

    total_liver_mm3 = 0

    for seg in SEGMENT_CLASSES:

        vol_mm3 = seg_stats.get(seg, {}).get("volume", 0)

        total_liver_mm3 += vol_mm3

        row[f"{seg}_vol_mL"] = round(vol_mm3 / 1000, 2)

    row["total_liver_vol_mL"] = round(total_liver_mm3 / 1000, 2)

    # Liver lesion statistics
    lesion_stats = task_results.get("liver_lesions") or {}

    lesion_total_mm3 = 0

    for key, val in lesion_stats.items():

        if "lesion" in key.lower():

            lesion_total_mm3 += val.get("volume", 0)

    row["lesion_total_vol_mL"] = round(lesion_total_mm3 / 1000, 2)
    row["lesion_segment"] = None

    rows.append(row)

summary_df = pd.DataFrame(rows)

summary_df

,class,case,liver_segment_1_vol_mL,liver_segment_2_vol_mL,liver_segment_3_vol_mL,liver_segment_4_vol_mL,liver_segment_5_vol_mL,liver_segment_6_vol_mL,liver_segment_7_vol_mL,liver_segment_8_vol_mL,total_liver_vol_mL,lesion_total_vol_mL,lesion_segment
0,abnormal_no_liver,AC4214d8b,46.99,120.99,110.08,117.36,239.06,96.35,222.37,186.36,1139.55,0.12,None
1,abnormal_no_liver,AC4215e80,94.33,262.26,133.07,224.24,341.99,276.84,322.42,478.98,2134.12,0.04,None
2,abnormal_no_liver,AC423c083,81.05,312.44,107.98,278.35,296.24,225.18,362.95,534.50,2198.68,2.22,None
3,liver_abnormality_no_lesion,AC423d3ee,97.26,399.60,213.31,306.23,181.11,287.78,278.33,398.93,2162.56,0.02,None
4,liver_abnormality_no_lesion,AC4241a63,109.02,287.71,233.98,309.37,431.08,296.99,582.13,558.14,2808.41,2.56,None
5,liver_abnormality_no_lesion,AC4244844,78.85,333.17,163.09,172.00,230.49,286.78,268.72,334.35,1867.46,3.35,None
6,liver_lesion,AC423e736,52.38,261.17,166.06,342.15,711.23,679.91,978.28,636.98,3828.16,361.18,None
7,liver_lesion,AC424042b,44.30,186.44,26.75,130.55,236.17,164.50,162.00,334.73,1285.44,5.83,None
8,liver_lesion,AC4244f0b,111.36,391.85,115.14,232.11,229.53,488.45,357.02,537.61,2463.08,59.34,None
9,normal,AC4216dc5,55.01,192.33,93.06,141.45,150.32,128.01,229.23,255.55,1244.96,0.00,None


In [ ]:
import nibabel as nib
import numpy as np

def get_lesion_segment(case_class, case_name):

    # Only liver lesion cases should have lesion segments
    if case_class != "liver_lesion":
        return None

    case_dir = OUTPUT_DIR / case_class / case_name

    seg_dir = case_dir / "liver_segments"
    lesion_dir = case_dir / "liver_lesions"

    lesion_files = list(lesion_dir.glob("*lesion*.nii.gz"))

    if len(lesion_files) == 0:
        return None

    lesion = nib.load(str(lesion_files[0])).get_fdata() > 0

    best_segment = None
    best_overlap = 0

    for i in range(1, 9):

        seg_file = seg_dir / f"liver_segment_{i}.nii.gz"

        if not seg_file.exists():
            continue

        seg = nib.load(str(seg_file)).get_fdata() > 0

        overlap = np.logical_and(seg, lesion).sum()

        if overlap > best_overlap:
            best_overlap = overlap
            best_segment = i

    if best_overlap == 0:
        return None

    return f"Segment {best_segment}"


summary_df["lesion_segment"] = summary_df.apply(
    lambda row: get_lesion_segment(row["class"], row["case"]),
    axis=1
)

summary_df.loc[
    summary_df["class"] != "liver_lesion",
    ["lesion_total_vol_mL", "lesion_segment"]
] = [0, None]

summary_df

,class,case,liver_segment_1_vol_mL,liver_segment_2_vol_mL,liver_segment_3_vol_mL,liver_segment_4_vol_mL,liver_segment_5_vol_mL,liver_segment_6_vol_mL,liver_segment_7_vol_mL,liver_segment_8_vol_mL,total_liver_vol_mL,lesion_total_vol_mL,lesion_segment
0,abnormal_no_liver,AC4214d8b,46.99,120.99,110.08,117.36,239.06,96.35,222.37,186.36,1139.55,0.00,None
1,abnormal_no_liver,AC4215e80,94.33,262.26,133.07,224.24,341.99,276.84,322.42,478.98,2134.12,0.00,None
2,abnormal_no_liver,AC423c083,81.05,312.44,107.98,278.35,296.24,225.18,362.95,534.50,2198.68,0.00,None
3,liver_abnormality_no_lesion,AC423d3ee,97.26,399.60,213.31,306.23,181.11,287.78,278.33,398.93,2162.56,0.00,None
4,liver_abnormality_no_lesion,AC4241a63,109.02,287.71,233.98,309.37,431.08,296.99,582.13,558.14,2808.41,0.00,None
5,liver_abnormality_no_lesion,AC4244844,78.85,333.17,163.09,172.00,230.49,286.78,268.72,334.35,1867.46,0.00,None
6,liver_lesion,AC423e736,52.38,261.17,166.06,342.15,711.23,679.91,978.28,636.98,3828.16,361.18,Segment 7
7,liver_lesion,AC424042b,44.30,186.44,26.75,130.55,236.17,164.50,162.00,334.73,1285.44,5.83,Segment 6
8,liver_lesion,AC4244f0b,111.36,391.85,115.14,232.11,229.53,488.45,357.02,537.61,2463.08,59.34,Segment 6
9,normal,AC4216dc5,55.01,192.33,93.06,141.45,150.32,128.01,229.23,255.55,1244.96,0.00,None


In [ ]:
# Save the summary table
summary_csv = OUTPUT_DIR / "summary_all_cases.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"Saved summary to: {summary_csv.resolve()}")

Saved summary to: C:\Users\HP EliteBook\Desktop\Digilians GP\liver_seg_outputs\summary_all_cases.csv


In [ ]:
# ==================================================
# Split cases based on zero volumes
# ==================================================

# Columns for the 8 liver segments
segment_volume_cols = [
    f"liver_segment_{i}_vol_mL"
    for i in range(1, 9)
]

# Check if ANY segment has zero volume
zero_segment = (
    summary_df[segment_volume_cols] == 0
).any(axis=1)

# Check total liver or lesion volume
zero_liver_or_lesion = (
    (summary_df["total_liver_vol_mL"] == 0) |
    (summary_df["lesion_total_vol_mL"] == 0)
)

# --------------------------------------------------
# Cases with ANY zero
# --------------------------------------------------

zero_df = summary_df[
    zero_liver_or_lesion | zero_segment
].copy()

# --------------------------------------------------
# Cases with NO zero anywhere
# --------------------------------------------------

non_zero_df = summary_df[
    ~(zero_liver_or_lesion | zero_segment)
].copy()


# ==================================================
# Save both CSV files
# ==================================================

zero_csv = OUTPUT_DIR / "cases_with_zero_volume.csv"
non_zero_csv = OUTPUT_DIR / "cases_without_zero_volume.csv"

zero_df.to_csv(zero_csv, index=False)
non_zero_df.to_csv(non_zero_csv, index=False)


# ==================================================
# Results
# ==================================================

print(f"Cases with zero volume    : {len(zero_df)}")
print(f"Cases without zero volume : {len(non_zero_df)}")

print("\nSaved:")
print(zero_csv)
print(non_zero_csv)